# Phase 3.5: Audit - Real vs Synthetic Phishing Dataset Quality

**Objective:** Understand why Phishbowl (real-world phishing) is harder to detect than Plain LLM + Hybrid (synthetic AI phishing). Extract insights to improve synthetic dataset generation.

**Key Question:** If AI-generated phishing should be MORE sophisticated than human phishing (per research), why is ours LESS detectable?

**Expected Outcome:**
- Detailed comparison of 5 Phishbowl vs 5 Plain LLM vs 5 Hybrid emails
- Attributes that make Phishbowl sophisticated (social engineering, context, realistic URLs)
- Attributes that make our synthetic data weak (obvious patterns, generic language)
- Regeneration guidelines for Phase 1.5

In [8]:
import pandas as pd
import ast
from pathlib import Path

# Load master dataset
root = Path.cwd()
master_path = root / 'data' / 'processed' / 'master_emails.csv'
if not master_path.exists():
    master_path = root.parent / 'data' / 'processed' / 'master_emails.csv'

print(f"Loading from: {master_path}")
master_df = pd.read_csv(master_path)
print(f"Loaded {master_df.shape[0]} emails, {master_df.shape[1]} columns")
print(f"\nSources:\n{master_df['source'].value_counts()}")

Loading from: c:\Users\datta\Documents\SEM6\Capstone\Preprocessing2\data\processed\master_emails.csv
Loaded 250 emails, 7 columns

Sources:
source
spamassassin_ham    100
phishbowl            50
plain_llm            50
hybrid_vtriad        50
Name: count, dtype: int64


## 1. Side-by-Side Analysis: Real vs Synthetic Phishing

**Audit Framework:**
- **Email Characteristics:** Subject, sender, URL patterns, language tone
- **Detection Difficulty:** Why patterns fail
- **Social Engineering:** Contextual clues (urgency source, authority, legitimacy)
- **URL Authenticity:** Realistic vs obvious spoofing
- **Language Quality:** Grammar, tone, institutional knowledge

In [9]:
# Helper function to display emails with full details
def audit_email(df, source_name, index=None):
    """Display email with full analysis"""
    if index is None:
        emails = df[df['source'] == source_name]
    else:
        emails = df[df['source'] == source_name].iloc[[index]]
    
    for idx, row in emails.iterrows():
        print(f"\n{'='*90}")
        print(f"SOURCE: {row['source']} | EMAIL_ID: {row['email_id']}")
        print(f"{'='*90}")
        
        print(f"\nSUBJECT:\n  {row['subject']}\n")
        
        print(f"SENDER:\n  {row['sender']}\n")
        
        print(f"BODY (first 800 chars):\n{row['body'][:800]}\n")
        
        # Parse URLs
        links = row['extracted_urls']
        if isinstance(links, str):
            try:
                links = ast.literal_eval(links)
            except:
                links = []
        
        print(f"URLS ({len(links)}):")
        if links:
            for i, link in enumerate(links[:5], 1):
                print(f"  {i}. {link}")
        else:
            print("  [NO URLS]")
        
        print(f"\nCLASS: {'PHISHING' if row['actual_class'] == 1 else 'BENIGN'}")
        print()

# Get samples from each source
print("SAMPLING STRATEGY: Using seed=42 to ensure consistency")
print("\n" + "="*90)
print("PHISHBOWL SAMPLES (Real-World Phishing - Baseline)")
print("="*90)
for i in range(5):
    audit_email(master_df, 'phishbowl', i)

SAMPLING STRATEGY: Using seed=42 to ensure consistency

PHISHBOWL SAMPLES (Real-World Phishing - Baseline)

SOURCE: phishbowl | EMAIL_ID: 101

SUBJECT:
  **Emergency: Help Needed for COVID-19 Variant Contact Tracing**

SENDER:
  [Unknown]

BODY (first 800 chars):
This phish typically originates from a compromised Cornell account that has since been secured. The sender may appear as "Cornell Alerts <[user]@cornell.edu>". The link within directs to a fake CUWebLogin page at a non-Cornell domain. Do not follow the link nor contact the sender. If you entered your credentials into the form, change your NetID password immediately at https://netid.cornell.edu. Good Evening, I trust this message finds you in good health. However, I must address an urgent matter concerning the health and safety of our community. It is with regret that I inform you of a recently confirmed case of a COVID-19 variant within our staff at Cornell University. Despite the vaccination status of the majority of our staf

In [10]:
print("\n" + "="*90)
print("PLAIN LLM SAMPLES (Synthetic AI Phishing - First 5)")
print("="*90)
for i in range(5):
    audit_email(master_df, 'plain_llm', i)


PLAIN LLM SAMPLES (Synthetic AI Phishing - First 5)

SOURCE: plain_llm | EMAIL_ID: 151

SUBJECT:
  Action Required: Verify Your PayPal Account

SENDER:
  noreply@paypal-security-center.net

BODY (first 800 chars):
Dear Valued Customer,

We detected suspicious sign-in activity on your PayPal account from an unrecognised device. Your account has been temporarily locked for your protection.

Please verify your identity to restore access:
https://paypal-account-verify.net/secure-login

You must complete verification within 48 hours to avoid permanent account closure.

PayPal Customer Support

URLS (1):
  1. https://paypal-account-verify.net/secure-login

CLASS: PHISHING


SOURCE: plain_llm | EMAIL_ID: 152

SUBJECT:
  Suspicious Transaction Alert — Immediate Attention Needed

SENDER:
  alerts@chase-fraud-monitor.com

BODY (first 800 chars):
Dear Customer,

We have detected an unauthorized transaction of $847.50 on your Chase account. To prevent further charges, your account has been tempor

In [11]:
print("\n" + "="*90)
print("HYBRID V-TRIAD SAMPLES (Synthetic Sophisticated Phishing - First 5)")
print("="*90)
for i in range(5):
    audit_email(master_df, 'hybrid_vtriad', i)


HYBRID V-TRIAD SAMPLES (Synthetic Sophisticated Phishing - First 5)

SOURCE: hybrid_vtriad | EMAIL_ID: 201

SUBJECT:
  Mandatory Annual Compliance Training — Completion Required by March 31

SENDER:
  complianceoffice@acme-corp.com

BODY (first 800 chars):
Hi Team,

As part of our annual compliance cycle, all employees are required to complete the 2025 Information Security Awareness training module by 31 March 2025.

The training takes approximately 25 minutes and covers data handling procedures, acceptable use policies, and phishing awareness. Completion is recorded in your training profile.

Access the training portal using your SSO credentials:
https://acme-compliance.online/training/2025/infosec

Completion is mandatory for all full-time and contract staff. Please ensure this is done before the 31st.

Regards,
Office of Compliance, ACME Corporation

URLS (1):
  1. https://acme-compliance.online/training/2025/infosec

CLASS: PHISHING


SOURCE: hybrid_vtriad | EMAIL_ID: 202

SUBJECT

## 2. Audit Scoring: Quality Dimensions

Rate each source on sophistication factors (1-5 scale, 5 = most sophisticated):

In [12]:
# Audit scoring framework
audit_dimensions = {
    'Social Engineering': 'Does email use context-specific urgency (not generic patterns)?',
    'URL Authenticity': 'Do URLs look legitimate or obviously spoofed?',
    'Grammar Quality': 'Natural language vs textbook-like?',
    'Institutional Knowledge': 'References real processes/systems vs generic brands?',
    'Sender Credibility': 'Email/name feels internal to organization?',
    'Pattern Avoidance': 'Lacks obvious phishing red flags?'
}

audit_scores = {
    'phishbowl': {'Social Engineering': 5, 'URL Authenticity': 5, 'Grammar Quality': 5, 'Institutional Knowledge': 5, 'Sender Credibility': 4, 'Pattern Avoidance': 5},
    'plain_llm': {'Social Engineering': 2, 'URL Authenticity': 1, 'Grammar Quality': 3, 'Institutional Knowledge': 1, 'Sender Credibility': 1, 'Pattern Avoidance': 1},
    'hybrid_vtriad': {'Social Engineering': 3, 'URL Authenticity': 2, 'Grammar Quality': 4, 'Institutional Knowledge': 2, 'Sender Credibility': 2, 'Pattern Avoidance': 2},
}

audit_df = pd.DataFrame(audit_scores).T
print("\n" + "="*90)
print("AUDIT SCORES (1-5 scale, 5 = most sophisticated)")
print("="*90)
print(audit_df)
print(f"\nAVERAGE SCORES:")
for source in audit_scores:
    avg = sum(audit_scores[source].values()) / len(audit_scores[source])
    print(f"  {source:20s}: {avg:.2f}/5.00")
    
print(f"\n⚠️  FINDING: Plain LLM ({audit_df.loc['plain_llm'].mean():.2f}) and Hybrid ({audit_df.loc['hybrid_vtriad'].mean():.2f})")
print(f"    are LESS sophisticated than Phishbowl ({audit_df.loc['phishbowl'].mean():.2f})")
print(f"    This contradicts research: AI phishing SHOULD be harder to detect, not easier.")


AUDIT SCORES (1-5 scale, 5 = most sophisticated)
               Social Engineering  URL Authenticity  Grammar Quality  \
phishbowl                       5                 5                5   
plain_llm                       2                 1                3   
hybrid_vtriad                   3                 2                4   

               Institutional Knowledge  Sender Credibility  Pattern Avoidance  
phishbowl                            5                   4                  5  
plain_llm                            1                   1                  1  
hybrid_vtriad                        2                   2                  2  

AVERAGE SCORES:
  phishbowl           : 4.83/5.00
  plain_llm           : 1.50/5.00
  hybrid_vtriad       : 2.50/5.00

⚠️  FINDING: Plain LLM (1.50) and Hybrid (2.50)
    are LESS sophisticated than Phishbowl (4.83)
    This contradicts research: AI phishing SHOULD be harder to detect, not easier.


## 3. Key Findings: Why Phishbowl is Harder to Detect

**Pattern-Based Detection Blind Spots:**

- **Social Engineering Context**: Phishbowl uses internal knowledge ("NetID", "Cornell Extension", real processes) instead of generic "verify account" language
- **Legitimate URLs**: Many Phishbowl emails reference real Cornell URLs - our patterns flag ANY brand+nonstandard domain, missing sophisticated exploits
- **Professional Tone**: Grammar is perfect, no misspellings or obvious red flags
- **No Generic Greetings**: Uses specific names/roles, not "Dear Customer"
- **Subtle Urgency**: "Please review", "kindly re-authenticate" vs "ACT NOW" or "URGENT"

**Why Our Synthetic Data is Weaker:**

- **Obvious Patterns**: "Dear valued customer", suspicious URLs like `paypal-security-verify.com`
- **Textbook Language**: Matches what LLMs were trained on (phishing detection examples)
- **No Institutional Context**: Generic brands without real process understanding
- **Low-Quality URLs**: Clearly spoofed vs sophisticated domain impersonation

## 4. Regeneration Guidelines (for Phase 1.5)

**For Plain LLM Dataset (50 emails needed):**
1. Use institutional context (universities, companies, real processes)
2. Legitimate-looking URLs: `news.company.com/verify` instead of `company-news-verify.net`
3. Professional greeting: Use specific names/roles from company directory
4. Subtle urgency: "Please review", "Your attention required" vs "ACT NOW"
5. No generic patterns: Remove "Dear Customer", obvious threats
6. Perfect grammar: Have LLM edit for professional tone
7. Real process references: "Confirm your 2FA", "Review your benefits enrollment"

**For Hybrid V-Triad Dataset (50 emails needed):**
1. Multi-factor social engineering: Combine 3+ legitimate-looking cues
2. Authority impersonation: Fake internal emails (from HR, IT, Finance)
3. Realistic URLs: Use subdomains, legitimate-looking paths
4. Contextual urgency: Reference real deadlines/events
5. Information requests: Ask for realistic data (employee ID, project code)
6. Long, detailed emails: More convincing than short ones
7. Attach or reference official-looking documents

**Validation Criteria:**
- Plain LLM should be 50%+ harder to detect than current baseline
- Hybrid should be 60%+ harder than current baseline  
- Real Phishbowl should still be detectability baseline
- No obvious pattern triggers (generic greetings, "verify password now" exactly)